# MDI 3003 - Advanced Predictive Analytics
`Sarah Blessy N [23MID0296]` 

`Akash M [23MID 0411]`

#  Kaggle Titanic – Machine Learning from Disaster

`Objective`
The objective of this project is to build a machine learning model that predicts whether a passenger survived the sinking of the Titanic based on passenger information such as age, gender, passenger class, fare, and family relationships.

`Goal`
To analyze the passenger dataset, identify patterns associated with survival, train and evaluate a classification model, and generate predictions for the passengers in the test dataset.

`Dataset Description`
- **Training Dataset (`train.csv`):** Contains passenger details and their actual survival status.
- **Testing Dataset (`test.csv`):** Contains passenger details without the survival labels.
- **Submission File (`submission.csv`):** Contains the predicted survival outcomes in the format required by Kaggle.

`Methodology`
1. Data loading and exploratory data analysis.
2. Data preprocessing and handling missing values.
3. Feature selection and encoding categorical variables.
4. Model training using the Random Forest Classifier.
5. Model evaluation using validation accuracy and a confusion matrix.
6. Generating predictions and preparing the Kaggle submission file.

`Evaluation Metric`
The model is evaluated using **accuracy**, which measures the proportion of correctly predicted survival outcomes.

`Expected Outcome`
A trained classification model that predicts passenger survival and generates a valid CSV file for submission to the Kaggle Titanic competition.

# Import libraries and load the datasets

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [6]:
train = pd.read_csv("C:/Users/Sarah Blessy/OneDrive/Desktop/FallSemester/C2 - Advanced Predictive Analytics/train.csv")
test = pd.read_csv("C:/Users/Sarah Blessy/OneDrive/Desktop/FallSemester/C2 - Advanced Predictive Analytics/test.csv")

In [7]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [9]:
print("Training data:", train.shape)
print("Testing data:", test.shape)

Training data: (891, 12)
Testing data: (418, 11)


# Explore the data

In [10]:
print("Missing values in training data:")
print(train.isnull().sum())

Missing values in training data:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [11]:
print("\nSurvival distribution:")
print(train["Survived"].value_counts())


Survival distribution:
Survived
0    549
1    342
Name: count, dtype: int64


In [12]:
print("\nSurvival rate by gender:")
print(train.groupby("Sex")["Survived"].mean())


Survival rate by gender:
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


In [13]:
print("\nSurvival rate by passenger class:")
print(train.groupby("Pclass")["Survived"].mean())


Survival rate by passenger class:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


# Select features and split the data

In [14]:
features = ["Pclass", "Sex", "Age", "SibSp","Parch", "Fare", "Embarked"]

In [15]:
X = train[features]
y = train["Survived"]

In [16]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

Training samples: 712
Validation samples: 179


`X contains the input features`



`y contains the target variable, Survived`



`80% of the data is used for training and 20% for validation`



`stratify=y preserves approximately the same survival-class proportions in both sets`

# Preprocess the data

In [18]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

In [19]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [20]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [21]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [22]:
print("Preprocessing pipeline created.")

Preprocessing pipeline created.


# Train the Random Forest model

In [23]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=5,
        random_state=42,
        class_weight="balanced"
    ))
])

In [24]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Evaluate the model

In [26]:
y_pred = model.predict(X_val)

In [27]:
accuracy = accuracy_score(y_val, y_pred)

In [28]:
print("Validation Accuracy:", round(accuracy * 100, 2), "%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Validation Accuracy: 79.33 %

Confusion Matrix:
[[94 16]
 [21 48]]


In [29]:
print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75      0.70      0.72        69

    accuracy                           0.79       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



# Train the final model on all training data

In [30]:
X_full = train[features]
y_full = train["Survived"]

In [31]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=5,
        random_state=42,
        class_weight="balanced"
    ))
])

In [32]:
final_model.fit(X_full, y_full)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Predict survival for the test dataset

In [33]:
X_test = test[features]

In [34]:
test_predictions = final_model.predict(X_test)

In [35]:
print("Number of predictions:", len(test_predictions))
print("First 10 predictions:", test_predictions[:10])

Number of predictions: 418
First 10 predictions: [0 0 0 0 1 0 0 0 1 0]


# Create the Kaggle submission CSV

In [36]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions.astype(int)
})

In [37]:
submission.to_csv("submission.csv", index=False)

In [38]:
print("Submission file created successfully!")
display(submission.head())

Submission file created successfully!


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [39]:
print("Submission shape:", submission.shape)
print("Column names:", submission.columns.tolist())
print("Missing predictions:", submission["Survived"].isnull().sum())

Submission shape: (418, 2)
Column names: ['PassengerId', 'Survived']
Missing predictions: 0
